# Phase 4 — Agent A2 : Préprocesseur
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre et valide l'agent A2 (`agents/a2_preprocessor.py`) :
- Normalisation du texte (minuscules, collapse espaces, strip)
- Filtrage par longueur (MIN=3, MAX=2000 chars)
- Déduplication exacte (après normalisation)
- Détection de langue via `langdetect`

> **Prérequis** : `pip install langdetect` — aucun LLM requis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from agents.a2_preprocessor import a2_preprocessor, MIN_CHARS, MAX_CHARS
from utils.text_cleaner import normalize_text

print(f"MIN_CHARS={MIN_CHARS}  MAX_CHARS={MAX_CHARS}")

## 1. normalize_text — Tests unitaires en direct

In [ ]:
tests_normalize = [
    ("HELLO World",   "hello world"),
    ("  strip me  ",  "strip me"),
    ("a  b   c",      "a b c"),
    ("line1\nline2",  "line1 line2"),
    (None,            None),
]

print(f"{'Input':<25} {'Expected':<20} {'Got':<20} {'OK'}")
print("-" * 75)
for inp, expected in tests_normalize:
    got = normalize_text(inp)
    ok = "✓" if got == expected else "✗"
    print(f"{str(inp):<25} {str(expected):<20} {str(got):<20} {ok}")

## 2. Exécution de A2 — corpus varié

In [ ]:
raw = [
    {"text": "HELLO World, this is a great video!"},
    {"text": "ab"},                           # trop court → filtré
    {"text": "Duplicate comment"},
    {"text": "Duplicate comment"},            # doublon → filtré
    {"text": "   HELLO WORLD, this is a great video!  "},  # doublon normalisé → filtré
    {"text": "Très belle vidéo, j'ai beaucoup appris!"},   # français
    {"text": ""},                             # vide → filtré
]

result = a2_preprocessor({"raw_comments": raw})
cleaned = result["cleaned_comments"]
print(f"Entrée : {len(raw)} | Sortie : {len(cleaned)} (filtrés / dédupliqués)")
print()
for c in cleaned:
    print(f"  lang={c['language']!r:5s}  cleaned_text={c['cleaned_text']!r}")

## 3. Statistiques post-prétraitement

In [ ]:
from collections import Counter

# Distribution des langues
langs = Counter(c["language"] for c in cleaned)
print("Distribution des langues :", dict(langs))

# Longueurs
lengths = [len(c["cleaned_text"]) for c in cleaned]
print(f"Longueur min/moy/max : {min(lengths)} / {sum(lengths)/len(lengths):.1f} / {max(lengths)}")

# Confirmation des critères
for c in cleaned:
    ct = c["cleaned_text"]
    assert len(ct) >= 3, f"Trop court : {ct!r}"
    assert len(ct) <= 2000, f"Trop long : {ct!r}"
print("Tous les commentaires respectent MIN_CHARS=3 et MAX_CHARS=2000.")

## Résumé A2

| Comportement | Résultat |
|---|---|
| Normalisation (lowercase + collapse) | Oui |
| Filtrage < 3 chars | Oui |
| Déduplication case-insensitive | Oui |
| Détection langue | Oui (champ `language`) |
| Métadonnées optionnelles préservées | Oui (`video_id`, `author_likes`, etc.) |